In [1]:
# ==========================================
# 1. IMPORTS & U-NET BLUEPRINTS
# ==========================================
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class UNetDown(nn.Module):
    def __init__(self, in_size, out_size, normalize=True, dropout=0.0):
        super(UNetDown, self).__init__()
        layers = [nn.Conv2d(in_size, out_size, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_size))
        layers.append(nn.LeakyReLU(0.2))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)
    def forward(self, x): return self.model(x)

class UNetUp(nn.Module):
    def __init__(self, in_size, out_size, dropout=0.0):
        super(UNetUp, self).__init__()
        layers = [
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(in_size, out_size, 3, 1, 1, bias=False),
            nn.InstanceNorm2d(out_size),
            nn.ReLU(inplace=True)
        ]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)
    def forward(self, x, skip_input):
        x = self.model(x)
        return torch.cat((x, skip_input), 1)

class NatraZ_UNet(nn.Module):
    def __init__(self):
        super(NatraZ_UNet, self).__init__()
        self.down1 = UNetDown(1, 64, normalize=False)
        self.down2 = UNetDown(64, 128)
        self.down3 = UNetDown(128, 256)
        self.down4 = UNetDown(256, 512, dropout=0.5)
        self.down5 = UNetDown(512, 512, dropout=0.5)
        self.up1 = UNetUp(512, 512, dropout=0.5)
        self.up2 = UNetUp(1024, 256)
        self.up3 = UNetUp(512, 128)
        self.up4 = UNetUp(256, 64)
        self.final = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(128, 3, 3, 1, 1),
            nn.Tanh()
        )
    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        d5 = self.down5(d4)
        u1 = self.up1(d5, d4)
        u2 = self.up2(u1, d3)
        u3 = self.up3(u2, d2)
        u4 = self.up4(u3, d1)
        return self.final(u4)

# ==========================================
# 2. THE LANDSAT DATA LOADER
# ==========================================
class LandsatDataset(Dataset):
    def __init__(self, base_path, transform=None):
        self.base_path = base_path
        self.transform = transform
        self.image_pairs = []
        
        rgb_dir = os.path.join(base_path, "output", "rgb_images")
        input_dir = os.path.join(base_path, "input")
        
        # Hunt down all the RGB targets
        if os.path.exists(rgb_dir):
            for rgb_file in os.listdir(rgb_dir):
                if not rgb_file.endswith(".tif"): continue
                
                # Extract the ID (e.g., "D-6" from "D-6_rgb_30m.tif")
                day_id = rgb_file.split("_")[0]
                
                # Construct the path to the Thermal B10 image
                thermal_file = os.path.join(input_dir, day_id, f"{day_id}_B10.tiff")
                rgb_full_path = os.path.join(rgb_dir, rgb_file)
                
                # Lock them in ONLY if both exist perfectly
                if os.path.exists(thermal_file) and os.path.exists(rgb_full_path):
                    self.image_pairs.append((thermal_file, rgb_full_path))
                    
        print(f"🌍 Landsat Protocol: Locked in {len(self.image_pairs)} perfect satellite pairs!")

    def __len__(self):
        return len(self.image_pairs)

    def __getitem__(self, idx):
        thermal_path, rgb_path = self.image_pairs[idx]
        
        # Convert TIFFs to standard Tensors
        thermal_image = Image.open(thermal_path).convert("L")
        rgb_image = Image.open(rgb_path).convert("RGB")
        
        if self.transform:
            thermal_image = self.transform(thermal_image)
            rgb_image = self.transform(rgb_image)
            
        return thermal_image, rgb_image

class GradientAlignmentLoss(nn.Module):
    def __init__(self):
        super(GradientAlignmentLoss, self).__init__()
        import numpy as np
        kernel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
        kernel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32)
        self.weight_x = nn.Parameter(torch.from_numpy(kernel_x).view(1, 1, 3, 3), requires_grad=False)
        self.weight_y = nn.Parameter(torch.from_numpy(kernel_y).view(1, 1, 3, 3), requires_grad=False)
        self.criterion = nn.L1Loss()
    def forward(self, gen_img, real_ir):
        gen_gray = 0.299 * gen_img[:, 0:1] + 0.587 * gen_img[:, 1:2] + 0.114 * gen_img[:, 2:3]
        gen_grad_x, gen_grad_y = F.conv2d(gen_gray, self.weight_x, padding=1), F.conv2d(gen_gray, self.weight_y, padding=1)
        ir_grad_x, ir_grad_y = F.conv2d(real_ir, self.weight_x, padding=1), F.conv2d(real_ir, self.weight_y, padding=1)
        return self.criterion(torch.abs(gen_grad_x) + torch.abs(gen_grad_y), torch.abs(ir_grad_x) + torch.abs(ir_grad_y))

# ==========================================
# 3. INITIALIZE THE NEW ENVIRONMENT
# ========================================== 
# 🚨 Confirming the correct Landsat Path:
LANDSAT_BASE = "/kaggle/input/datasets/vinaykumarsingh2712/landset-9-and-8-dataset-for-band-b2b3b4b10/DATASET"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🚀 U-Net Initialization Sequence on: {device} 🚀")

generator = NatraZ_UNet().to(device)
print("🧠 Fresh Brain Generated. Ready for Satellite Data.")

# ==========================================
# 4. THE SATELLITE TRAINING LOOP
# ========================================== 
def train_unet_landsat(epochs, batch_size, base_path, gen_model):
    transform = transforms.Compose([
        transforms.Resize((256, 256)), 
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])

    dataset = LandsatDataset(base_path=base_path, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    optimizer = optim.Adam(gen_model.parameters(), lr=0.0002, betas=(0.5, 0.999))
    l1_loss = nn.L1Loss()
    grad_loss = GradientAlignmentLoss().to(device)

    gen_model.train()

    for epoch in range(epochs):
        for i, (ir_imgs, rgb_imgs) in enumerate(dataloader):
            ir_imgs, rgb_imgs = ir_imgs.to(device), rgb_imgs.to(device)

            optimizer.zero_grad()
            generated_rgb = gen_model(ir_imgs)
            loss_color = l1_loss(generated_rgb, rgb_imgs)
            loss_edge = grad_loss(generated_rgb, ir_imgs)

            total_loss = loss_color + (0.5 * loss_edge)
            total_loss.backward()
            optimizer.step()

        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {total_loss.item():.4f}")
        torch.save(gen_model.state_dict(), "/kaggle/working/NatraZ_UNet_Landsat.pth")

    print("\n✅ U-Net Training Complete!")
    return gen_model

# 5. LAUNCH THE 1-EPOCH TEST
print("\nInitiating 1-Epoch Safety Test...")
trained_unet_final = train_unet_landsat(epochs=450, batch_size=4, base_path=LANDSAT_BASE, gen_model=generator)


🚀 U-Net Initialization Sequence on: cuda 🚀
🧠 Fresh Brain Generated. Ready for Satellite Data.

Initiating 1-Epoch Safety Test...
🌍 Landsat Protocol: Locked in 45 perfect satellite pairs!
Epoch [1/450] | Loss: 0.2847
Epoch [2/450] | Loss: 0.2262
Epoch [3/450] | Loss: 0.2002
Epoch [4/450] | Loss: 0.1877
Epoch [5/450] | Loss: 0.1837
Epoch [6/450] | Loss: 0.1936
Epoch [7/450] | Loss: 0.2228
Epoch [8/450] | Loss: 0.2087
Epoch [9/450] | Loss: 0.1766
Epoch [10/450] | Loss: 0.1717
Epoch [11/450] | Loss: 0.1552
Epoch [12/450] | Loss: 0.1636
Epoch [13/450] | Loss: 0.1706
Epoch [14/450] | Loss: 0.2420
Epoch [15/450] | Loss: 0.1533
Epoch [16/450] | Loss: 0.2079
Epoch [17/450] | Loss: 0.1514
Epoch [18/450] | Loss: 0.1636
Epoch [19/450] | Loss: 0.1607
Epoch [20/450] | Loss: 0.1650
Epoch [21/450] | Loss: 0.1555
Epoch [22/450] | Loss: 0.1733
Epoch [23/450] | Loss: 0.2218
Epoch [24/450] | Loss: 0.2226
Epoch [25/450] | Loss: 0.1674
Epoch [26/450] | Loss: 0.1468
Epoch [27/450] | Loss: 0.3070
Epoch [28/4

In [2]:
# import os
# import torch
# import torch.nn as nn
# import matplotlib.pyplot as plt
# import random
# from torchvision import transforms
# from PIL import Image

# # ==========================================
# # 1. THE U-NET BLUEPRINTS (Required for Loading)
# # ==========================================
# class UNetDown(nn.Module):
#     def __init__(self, in_size, out_size, normalize=True, dropout=0.0):
#         super(UNetDown, self).__init__()
#         layers = [nn.Conv2d(in_size, out_size, 4, 2, 1, bias=False)]
#         if normalize:
#             layers.append(nn.InstanceNorm2d(out_size))
#         layers.append(nn.LeakyReLU(0.2))
#         if dropout:
#             layers.append(nn.Dropout(dropout))
#         self.model = nn.Sequential(*layers)
#     def forward(self, x): return self.model(x)

# class UNetUp(nn.Module):
#     def __init__(self, in_size, out_size, dropout=0.0):
#         super(UNetUp, self).__init__()
#         layers = [
#             nn.Upsample(scale_factor=2, mode='nearest'),
#             nn.Conv2d(in_size, out_size, 3, 1, 1, bias=False),
#             nn.InstanceNorm2d(out_size),
#             nn.ReLU(inplace=True)
#         ]
#         if dropout:
#             layers.append(nn.Dropout(dropout))
#         self.model = nn.Sequential(*layers)
#     def forward(self, x, skip_input):
#         x = self.model(x)
#         return torch.cat((x, skip_input), 1)

# class VisionZ_UNet(nn.Module):
#     def __init__(self):
#         super(VisionZ_UNet, self).__init__()
#         self.down1 = UNetDown(1, 64, normalize=False)
#         self.down2 = UNetDown(64, 128)
#         self.down3 = UNetDown(128, 256)
#         self.down4 = UNetDown(256, 512, dropout=0.5)
#         self.down5 = UNetDown(512, 512, dropout=0.5)
#         self.up1 = UNetUp(512, 512, dropout=0.5)
#         self.up2 = UNetUp(1024, 256)
#         self.up3 = UNetUp(512, 128)
#         self.up4 = UNetUp(256, 64)
#         self.final = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='nearest'),
#             nn.Conv2d(128, 3, 3, 1, 1),
#             nn.Tanh()
#         )
#     def forward(self, x):
#         d1 = self.down1(x)
#         d2 = self.down2(d1)
#         d3 = self.down3(d2)
#         d4 = self.down4(d3)
#         d5 = self.down5(d4)
#         u1 = self.up1(d5, d4)
#         u2 = self.up2(u1, d3)
#         u3 = self.up3(u2, d2)
#         u4 = self.up4(u3, d1)
#         return self.final(u4)

# # ==========================================
# # 2. PATHS & DEVICE SETUP
# # ==========================================
# BASE_PATH = "/kaggle/input/datasets/samdazel/teledyne-flir-adas-thermal-dataset-v2/FLIR_ADAS_v2"
# IR_FOLDER = os.path.join(BASE_PATH, 'images_thermal_train', 'data') 
# RGB_FOLDER = os.path.join(BASE_PATH, 'images_rgb_train', 'data') 

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"🚀 Visualizer Initialized on: {device} 🚀")

# # 🚨 PASTE YOUR COPIED DATASET PATH HERE:
# CHECKPOINT_PATH = "/kaggle/input/datasets/vishnukarthik678/unet-50-brain/VisionZ_UNet_50_Epoch.pth"

# # ==========================================
# # 3. LOAD MEMORIES & RENDER
# # ==========================================
# print("🧠 Retrieving 50-Epoch U-Net Brain...")
# generator = VisionZ_UNet().to(device)

# # Load the weights into the blueprint
# generator.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
# generator.eval()
# print("--> Memory loaded. Rendering 3x3 visual grid...")

# all_ir_files = sorted(os.listdir(IR_FOLDER))
# all_rgb_files = sorted(os.listdir(RGB_FOLDER))

# # Grab 3 random images
# indices = random.sample(range(len(all_ir_files)), 3)
# fig, axes = plt.subplots(3, 3, figsize=(12, 12))

# transform = transforms.Compose([
#     transforms.Resize((256, 256)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.5], std=[0.5])
# ])

# with torch.no_grad():
#     for row_idx, idx in enumerate(indices):
#         ir_path = os.path.join(IR_FOLDER, all_ir_files[idx])
#         rgb_path = os.path.join(RGB_FOLDER, all_rgb_files[idx])
        
#         ir_img = Image.open(ir_path).convert("L")
#         real_rgb = Image.open(rgb_path).convert("RGB")
        
#         ir_tensor = transform(ir_img).unsqueeze(0).to(device)
#         fake_rgb_tensor = generator(ir_tensor)
        
#         # De-normalize to display properly
#         fake_rgb = fake_rgb_tensor.squeeze().cpu()
#         fake_rgb = (fake_rgb * 0.5) + 0.5 
#         fake_rgb = fake_rgb.permute(1, 2, 0).numpy() 
        
#         axes[row_idx, 0].imshow(ir_img, cmap='gray')
#         axes[row_idx, 0].set_title("Input Thermal (IR)")
#         axes[row_idx, 0].axis('off')
        
#         axes[row_idx, 1].imshow(fake_rgb)
#         axes[row_idx, 1].set_title("AI Generated RGB (50 Epochs)")
#         axes[row_idx, 1].axis('off')
        
#         axes[row_idx, 2].imshow(real_rgb)
#         axes[row_idx, 2].set_title("Real Ground Truth")
#         axes[row_idx, 2].axis('off')

# plt.tight_layout()
# plt.show()

In [3]:
# #for know the file name 
# import os

# BASE_PATH = "/kaggle/input/datasets/samdazel/teledyne-flir-adas-thermal-dataset-v2/FLIR_ADAS_v2"
# IR_FOLDER = os.path.join(BASE_PATH, 'images_thermal_train', 'data') 
# RGB_FOLDER = os.path.join(BASE_PATH, 'images_rgb_train', 'data') 

# print("🔥 First 5 Thermal Files:")
# for f in sorted(os.listdir(IR_FOLDER))[:5]: print(f)

# print("\n📸 First 5 RGB Files:")
# for f in sorted(os.listdir(RGB_FOLDER))[:5]: print(f)

In [4]:
# #for getting the file names doing x ray
# import os

# BASE_PATH = "/kaggle/input/datasets/vinaykumarsingh2712/landset-9-and-8-dataset-for-band-b2b3b4b10/DATASET"

# print("🔥 Hunting for the ACTUAL Thermal files...")
# input_files_found = 0
# for root, dirs, files in os.walk(os.path.join(BASE_PATH, "input")):
#     for f in files:
#         print(os.path.join(root, f))
#         input_files_found += 1
#         if input_files_found >= 5: break
#     if input_files_found >= 5: break

# print("\n📸 Hunting for the ACTUAL RGB files...")
# output_files_found = 0
# for root, dirs, files in os.walk(os.path.join(BASE_PATH, "output")):
#     for f in files:
#         print(os.path.join(root, f))
#         output_files_found += 1
#         if output_files_found >= 5: break
#     if output_files_found >= 5: break